# 약관비교 - "쪼개지 않고 문맥 정리" 방식 테스트 노트북

기존 방식(`terms_verification_test.ipynb`)은 `value`에 여러 조건이 섞여 있으면 **분해(split)**해서 조건별로 각각 검증합니다.

이 노트북은 그 대신, `value`를 쪼개지 않고 **내용/의미는 그대로 둔 채 문장 구조·표현만 정리(재구성)**한
뒤 그 정제된 값 하나로 약관과 비교하는 대안을 테스트합니다. 정제 단계에서 내용이 추가되거나 빠지면 안 됩니다.

운영 코드(`app/services/terms_verification_service.py`)의 **최신** `SYSTEM_PROMPT`/`ITEM_VERIFICATION_SCHEMA`를
그대로 옮겨왔고(matchRate 포함), 분해 단계만 "정제" 단계로 교체했습니다.

## 사용 전 준비
1. 바로 아래 "설정값" 셀에 Azure OpenAI 자격증명을 직접 입력하세요.
2. 이미 OCR 처리된 약관 문서의 결과 텍스트 파일 경로를 입력하세요.
3. "검증 대상 데이터" 셀에서 상품명/항목(`itemNm`/`value`/`desc`)을 원하는 대로 수정해서 테스트하세요.
4. 마지막 섹션에서 실제 `report_service.build_report_xlsx()`로 리포트(.xlsx)를 생성합니다.

> 운영 코드의 프롬프트/스키마가 나중에 바뀌면, 이 노트북의 해당 셀도 같이 맞춰줘야 합니다.


In [ ]:
# ── 설정값 (직접 입력) ──────────────────────────────────────────
AZURE_OPENAI_ENDPOINT = "https://<your-resource-name>.openai.azure.com/"
AZURE_OPENAI_API_KEY = "<your-api-key>"
AZURE_OPENAI_DEPLOYMENT_NAME = "<deployment-name>"
AZURE_OPENAI_API_VERSION = "2024-12-01-preview"

# 이미 OCR 처리된 약관 문서의 결과 텍스트 파일 경로
# (parse-di의 result.md, 또는 document_intelligence_test.ipynb의 content.md 등)
OCR_RESULT_PATH = r"C:\path\to\result.md"

# 실제 /api/v1/terms/verify 요청과 동일한 구조의 JSON 파일 경로. data[]를 여기서 불러온다.
# knwlgNm/termInfo[0].termNm/termInfo[0].aplyDate도 있으면 그대로 가져와서 쓴다(없으면 아래 기본값 사용).
TEST_REQUEST_JSON_PATH = r"C:\Users\소영둥이\Desktop\azure-doc-ai-service\notebooks\sample_terms_verify_request.json"

# 위 JSON 파일에 knwlgNm/termNm/aplyDate가 없을 때 쓰일 기본값
KNWLG_NM = "테스트 지식"
TERM_NM = "테스트 약관"
APLY_DATE = "2026-01-01"

# 이 저장소 루트 경로 - app.schemas.terms / app.services.report_service를 import하기 위해 필요
PROJECT_ROOT = r"C:\Users\소영둥이\Desktop\azure-doc-ai-service"
# ─────────────────────────────────────────────────────────────


## 1. Azure OpenAI 클라이언트 준비 및 약관 문서 로드

In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

with open(OCR_RESULT_PATH, encoding="utf-8") as f:
    document_text = f.read()

print(f"문서 길이: {len(document_text)}자")
print(document_text[:300])


## 2. 검증 대상 데이터

`TEST_REQUEST_JSON_PATH`(실제 `/api/v1/terms/verify` 요청과 동일한 구조의 JSON 파일)에서 `data[]`를 그대로 불러옵니다.
`knwlgNm`/`termInfo[0].termNm`/`termInfo[0].aplyDate`도 파일에 있으면 그대로 가져와서 설정값을 덮어씁니다.

`data[]`는 `[{name, items:[{itemNm, value, desc}]}, ...]` 형태이며, 여러 name을 한 번에 담을 수 있습니다.
`value`에 여러 조건이 섞여 있어도 **쪼개지 않고** 그대로 하나의 항목으로 유지한 채, 정제 단계만 거칩니다.


In [ ]:
import json as _json

with open(TEST_REQUEST_JSON_PATH, encoding="utf-8") as f:
    request_json = _json.load(f)

# 파일에 있으면 그대로 쓰고, 없으면 위 설정값 기본값을 유지한다.
KNWLG_NM = request_json.get("knwlgNm", KNWLG_NM)
if request_json.get("termInfo"):
    TERM_NM = request_json["termInfo"][0].get("termNm", TERM_NM)
    APLY_DATE = request_json["termInfo"][0].get("aplyDate", APLY_DATE)

test_names = request_json["data"]  # [{name, items:[{itemNm, value, desc}]}, ...]

print(f"knwlgNm={KNWLG_NM!r}, termNm={TERM_NM!r}, aplyDate={APLY_DATE!r}")
print(f"불러온 name 그룹: {len(test_names)}개")
for group in test_names:
    print(f"  - {group['name']}: 항목 {len(group['items'])}개")


## 3. 프롬프트/스키마

`ITEM_VERIFICATION_SCHEMA`/`SYSTEM_PROMPT`/`_build_user_content`는 운영 코드 최신 버전을 그대로 옮겨왔습니다.

`VALUE_REFINE_*`가 이번에 새로 추가된 부분입니다 — "쪼개기(split)" 대신 "정제(refine)"만 수행하고,
결과는 항상 **문자열 하나**입니다(배열이 아님).


In [ ]:
SYSTEM_PROMPT = '당신은 약관 문서를 기준으로 상품 항목 데이터를 검증하는 어시스턴트입니다. 반드시 주어진 약관 원문 내용만을 근거로 판단하고, 원문에 없는 내용은 추측하지 마세요.\nitemNm이나 값이 약관 원문에 완전히 동일한 표현으로 나오지 않아도 괜찮습니다. 동의어·유사 표현·어순 차이 등으로 표현만 다를 뿐 의미가 같다면 관련 내용/일치하는 것으로 인정하세요. \'일치\'는 문자 그대로 같은 표현인지가 아니라 의미가 같은지를 기준으로 판단하세요.\n\n아래 순서대로 하나씩 판단해서 필드를 채우세요. 뒤 단계는 앞 단계에서 채운 내용을 근거로 판단하세요.\n\n1. llmValue: 약관 원문 전체(일반 원칙과 예외/단서 조항 모두 포함)를 검토해서 이 항목의 실제 값을 판단해 채우세요.\n   - 항목 설명(desc)이 있으면, 항목명(itemNm)이 정확히 무엇을 의미하는지 파악하는 데 참고하세요.\n   - 현재 값이 없으면, 상품명+항목명을 기준으로 약관에서 값을 찾아 채우세요.\n   - 약관에 이 항목에 대한 내용 자체가 없으면 null로 하세요.\n2. evidence: 위에서 판단한 llmValue의 근거가 되는 문장을 약관 원문에서 생략·의역 없이 그대로 인용하세요. 근거가 되는 문장이 여러 곳에 있으면, llmValue를 가장 직접적으로 뒷받침하는 문장 하나만 인용하세요. llmValue가 null이면 evidence도 null로 하세요.\n3. page: evidence가 위치한 페이지 번호를 원문의 <!-- PageNumber="N" --> 마커로 판단하세요. 이 마커는 각 페이지 본문의 앞이 아니라 뒤(하단, PageFooter 바로 다음)에 붙어 있으므로, evidence와 그 뒤에 처음 나오는 <!-- PageBreak --> 사이에 있는 PageNumber 마커의 값을 사용하세요 (evidence보다 앞에 나온 마커는 이전 페이지의 것이니 사용하지 마세요). 그 사이에 PageNumber 마커가 없으면 - 예를 들어 PageBreak가 먼저 나오는 경우 - 다음 페이지의 PageNumber를 가져다 쓰지 말고 null로 하세요. evidence가 null이면 page도 null로 하세요.\n4. article: evidence가 위치한 약관 조항 번호(예: "제3조", "제3조 2항")가 원문에 표기되어 있으면 기입하세요. evidence가 null이거나 조항을 특정할 수 없으면 null로 하세요.\n5. reason: 현재 값과 llmValue가 다른 경우 그 차이를 설명하세요. 현재 값과 llmValue가 완전히 같으면 null로 하세요.\n   - 차이가 있다면 그 차이가 다음 중 무엇인지 구분해서 명시하세요: ① 현재 값에 llmValue와 다른(틀린) 내용이 있는 경우 → 어떤 부분이 근거상 확인되지 않는/틀린 내용인지 명시하세요. ② llmValue에 있는 내용이 현재 값에서 빠진(누락된) 경우 → 어떤 내용이 빠졌는지 명시하세요. 두 가지가 함께 있다면 둘 다 명시하세요. \'누락되었습니다\' 같은 표현은 실제로 빠진 내용에 대해서만 쓰고, 현재 값 자체가 틀린 경우에는 \'틀린 내용입니다/근거에서 확인되지 않습니다\' 등으로 명확히 구분해서 표현하세요.\n6. matchRate: reason이 null이거나(현재 값과 llmValue가 완전히 일치) llmValue가 null(비교 대상 자체가 없음)이면 matchRate도 null로 하세요. 그 외의 경우, 현재 값이 llmValue와 의미적으로 얼마나 일치하는지 0~100 사이의 정수(일치율)로 판단하세요 - 완전히 무관한 내용이면 0에 가깝게, 사소한 차이(디테일 하나만 다름)면 100에 가깝게 판단하세요.\n7. status: 1~6에서 채운 내용을 종합해 다음 기준으로 최종 판정하세요. 세 상태는 서로 겹치지 않아야 합니다.\n   - MATCHED: 현재 값이 llmValue와 완전히 일치 (틀린 내용도 없고 빠진 내용도 없음)\n   - PARTIAL_MATCH: 현재 값에 llmValue와 다른(틀린) 내용은 없지만, llmValue에 있는 내용 중 일부가 현재 값에 빠져 있음 (현재 값에 포함된 내용 자체는 모두 맞음)\n   - MISMATCH: 다음 중 하나 — (a) 현재 값에 llmValue와 다른(틀린) 내용이 하나라도 있음, (b) 현재 값 자체가 없음, (c) llmValue가 null(약관에 이 항목에 대한 내용 자체가 없음)'

ITEM_VERIFICATION_SCHEMA = {
    "name": "terms_item_verification",
    "schema": {
        "type": "object",
        "properties": {
            "llmValue": {"type": ["string", "null"]},
            "evidence": {"type": ["string", "null"]},
            "page": {
                "type": ["integer", "null"],
                "description": "evidence가 위치한 페이지 번호. evidence와 그 뒤 첫 PageBreak 사이에 있는 PageNumber 마커 값 (마커는 각 페이지 본문 하단에 위치). 그 사이에 없으면 null",
            },
            "article": {
                "type": ["string", "null"],
                "description": "evidence가 위치한 약관 조항 번호, 예: '제3조', '제3조 2항'. 특정할 수 없으면 null",
            },
            "reason": {"type": ["string", "null"]},
            "matchRate": {
                "type": ["integer", "null"],
                "description": "현재 값과 llmValue의 의미적 일치율 (0~100). reason이 null이거나 llmValue가 null이면 null",
            },
            "status": {"type": "string", "enum": ["MATCHED", "PARTIAL_MATCH", "MISMATCH"]},
        },
        "required": ["llmValue", "evidence", "page", "article", "reason", "matchRate", "status"],
        "additionalProperties": False,
    },
    "strict": True,
}


def build_user_content(document_text, name, item_nm, desc, value):
    value_section = f'현재 값: "{value}"' if value else "현재 값: (없음, 약관에서 추출 필요)"
    return [
        {
            "type": "text",
            "text": f"[약관 원문]\n{document_text}\n\n",
            "prompt_cache_breakpoint": {"mode": "explicit"},
        },
        {
            "type": "text",
            "text": (
                f"[검증 대상]\n"
                f"상품명: {name}\n"
                f"항목명: {item_nm}\n"
                f"항목 설명: {desc or '(없음)'}\n"
                f"{value_section}"
            ),
        },
    ]


# ── 여기부터 "정제" 단계 (분해 대신) ──────────────────────────────
VALUE_REFINE_SYSTEM_PROMPT = (
    "당신은 상품 항목의 현재 값(value) 문장을 약관과 대조하기 좋은 형태로 다듬는 어시스턴트입니다. "
    "값에 담긴 내용/의미는 절대 추가·삭제·왜곡하지 말고, 문장 구조와 표현만 자연스럽게 정리해서 "
    "하나의 명확한 문장(또는 여러 조건이 있으면 자연스럽게 이어지는 하나의 문단)으로 재구성하세요. "
    "여러 조건이 나열식/개조식으로 섞여 있어도 쪼개지 말고 하나의 값으로 유지하세요."
)

VALUE_REFINE_SCHEMA = {
    "name": "value_refine",
    "schema": {
        "type": "object",
        "properties": {
            "refinedValue": {"type": "string"},
        },
        "required": ["refinedValue"],
        "additionalProperties": False,
    },
    "strict": True,
}


def build_refine_prompt(item_nm, desc, value):
    return (
        f"[정제 대상]\n항목명: {item_nm}\n항목 설명: {desc or '(없음)'}\n값: {value}\n\n"
        "위 값의 내용과 의미는 절대 바꾸지 말고(추가/누락 금지), 문장 구조와 표현만 자연스럽게 다듬어 "
        "하나의 명확한 문장(또는 문단)으로 재구성하세요. 쪼개서 여러 개로 반환하지 마세요."
    )


## 4. 호출 함수 (정제 + 검증)

호출마다 토큰 수(캐시 적중 포함)와 소요시간을 같이 출력합니다.


In [ ]:
import json
import time


def call_structured(system_prompt, user_content, json_schema):
    start = time.monotonic()
    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        response_format={"type": "json_schema", "json_schema": json_schema},
    )
    elapsed_ms = (time.monotonic() - start) * 1000
    usage = response.usage
    cached_tokens = 0
    if usage is not None and usage.prompt_tokens_details is not None:
        cached_tokens = usage.prompt_tokens_details.cached_tokens or 0

    return {
        "parsed": json.loads(response.choices[0].message.content),
        "model": response.model,
        "prompt_tokens": usage.prompt_tokens if usage else 0,
        "completion_tokens": usage.completion_tokens if usage else 0,
        "cached_tokens": cached_tokens,
        "elapsed_ms": elapsed_ms,
    }


def refine_value(item_nm, desc, value):
    """value가 없으면 정제 대상이 아니므로 그대로 반환한다."""
    if value is None:
        return None
    result = call_structured(
        VALUE_REFINE_SYSTEM_PROMPT, build_refine_prompt(item_nm, desc, value), VALUE_REFINE_SCHEMA
    )
    refined = result["parsed"]["refinedValue"]
    print(
        f"  [정제] {item_nm}: {result['prompt_tokens']}+{result['completion_tokens']}토큰, "
        f"{result['elapsed_ms']:.0f}ms"
    )
    print(f"    원본: {value!r}")
    print(f"    정제: {refined!r}")
    return refined


def verify_value(name, item_nm, desc, original_value, refined_value):
    result = call_structured(
        SYSTEM_PROMPT,
        build_user_content(document_text, name, item_nm, desc, refined_value),
        ITEM_VERIFICATION_SCHEMA,
    )
    parsed = result["parsed"]
    print(
        f"  [검증] {item_nm} -> {parsed['status']}, matchRate={parsed.get('matchRate')}, "
        f"{result['prompt_tokens']}+{result['completion_tokens']}토큰(캐시 {result['cached_tokens']}), "
        f"{result['elapsed_ms']:.0f}ms"
    )
    return {
        "itemNm": item_nm,
        "value": original_value,
        "subClaim": refined_value if refined_value != original_value else None,
        "status": parsed["status"],
        "llmValue": parsed.get("llmValue"),
        "evidence": parsed.get("evidence"),
        "page": parsed.get("page"),
        "article": parsed.get("article"),
        "reason": parsed.get("reason"),
        "matchRate": parsed.get("matchRate"),
    }


## 5. 전체 실행 (정제 → 검증, 쪼개지 않음)

In [ ]:
all_results_by_name = {}

for group in test_names:
    name = group["name"]
    print(f"\n########## {name} ##########")
    group_results = []
    for item in group["items"]:
        item_nm = item["itemNm"]
        value = item.get("value")
        desc = item.get("desc")
        print(f"\n=== {item_nm} (value={value!r}) ===")

        refined_value = refine_value(item_nm, desc, value)
        result = verify_value(name, item_nm, desc, value, refined_value)
        group_results.append(result)
    all_results_by_name[name] = group_results

print("\n\n=== 최종 결과 ===")
print(json.dumps(all_results_by_name, ensure_ascii=False, indent=2))


## 6. 리포트(.xlsx) 생성

실제 `app/services/report_service.py`의 `build_report_xlsx()`를 그대로 사용합니다
(`app.schemas.terms`/`app.services.report_service`는 `app.core.config`에 의존하지 않아서,
`.env` 없이도 이 노트북에서 바로 import해서 쓸 수 있습니다).


In [ ]:
import sys
from datetime import datetime

sys.path.insert(0, PROJECT_ROOT)

from app.schemas.terms import TermsDocumentItemsResult, TermsItemResult, TermsNameResult, TermsVerificationResult
from app.services import report_service

name_results = [
    TermsNameResult(
        name=name,
        documents=[
            TermsDocumentItemsResult(
                ocrResltKey="notebook-test",
                termNm=TERM_NM,
                aplyDate=APLY_DATE,
                items=[TermsItemResult(**r) for r in group_results],
            )
        ],
    )
    for name, group_results in all_results_by_name.items()
]

result = TermsVerificationResult(knwlgNm=KNWLG_NM, data=name_results)

verified_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
xlsx_bytes = report_service.build_report_xlsx(result, verified_at)

out_path = r"C:\Users\소영둥이\Desktop\report_refine_test.xlsx"
with open(out_path, "wb") as f:
    f.write(xlsx_bytes)

print(f"저장 완료: {out_path}")
